<a href="https://colab.research.google.com/github/agg506u/Plasmid-Copy-Number-Calculator/blob/main/%E3%83%97%E3%83%A9%E3%82%B9%E3%83%9F%E3%83%89DNA%E3%82%B3%E3%83%94%E3%83%BC%E6%95%B0%E8%A8%88%E7%AE%97%E6%A9%9F_ver2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
# Circular dsDNA（プラスミド）コピー数計算ツール
# 配列組成を反映・末端なし（circular）を考慮

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ===== CSS（ダーク・ライト両対応）=====
custom_css = """
<style>
button.widget-button {
    background-color: #2c7be5 !important;
    color: white !important;
    font-weight: bold;
    border: none !important;
}
</style>
"""

# ===== 定数 =====
# DNA中のデオキシヌクレオチド（monophosphate, g/mol）
base_weights = {
    'A': 331.2218,
    'T': 322.2085,
    'G': 347.2212,
    'C': 307.1971
}

WATER = 18.01528  # g/mol
AVOGADRO = 6.02214076e23

# ===== 入力ウィジェット =====
sequence_input = widgets.Textarea(
    value='',
    placeholder='ここにプラスミド配列（5\'→3\'、片鎖分）を貼り付けてください',
    description='配列:',
    layout=widgets.Layout(width='100%', height='160px'),
    style={'description_width': '60px'}
)

concentration_input = widgets.FloatText(
    value=0.0,
    description='ng/µL:',
    style={'description_width': '60px'}
)

output_area = widgets.Output(
    layout=widgets.Layout(width='100%', height='220px', border='1px solid #ccc')
)

# ===== 計算関数（Circular dsDNA）=====
def calculate_copy_number_circular(seq, conc_ng_uL):
    seq = seq.replace('\n', '').replace(' ', '').upper()

    if len(seq) == 0:
        raise ValueError("配列が空です")

    # 塩基カウント
    counts = {b: seq.count(b) for b in "ATGC"}
    length = sum(counts.values())  # bp

    # 1本鎖分子量（塩基合計）
    mw_ss = sum(counts[b] * base_weights[b] for b in counts)

    # Circular DNA：脱水縮合は N 回
    mw_ss -= WATER * length

    # dsDNA
    mw_ds = mw_ss * 2

    # コピー数計算
    g_per_uL = conc_ng_uL * 1e-9
    moles_per_uL = g_per_uL / mw_ds
    copies_per_uL = moles_per_uL * AVOGADRO

    return {
        "length_bp": length,
        "counts": counts,
        "mw_ds": mw_ds,
        "conc": conc_ng_uL,
        "copies": copies_per_uL
    }

# ===== ボタン =====
run_button = widgets.Button(
    description="計算",
    layout=widgets.Layout(width='120px', height='40px')
)

clear_button = widgets.Button(
    description="クリア",
    layout=widgets.Layout(width='120px', height='40px'),
    button_style='danger'
)

# ===== ボタン動作 =====
def on_run_clicked(b):
    with output_area:
        clear_output()
        try:
            result = calculate_copy_number_circular(
                sequence_input.value,
                concentration_input.value
            )

            print(f"""🧬 Circular dsDNA（プラスミド）コピー数計算結果

📏 長さ: {result['length_bp']} bp
🧩 塩基組成:
    A: {result['counts']['A']}
    T: {result['counts']['T']}
    G: {result['counts']['G']}
    C: {result['counts']['C']}

⚖️ 分子量（dsDNA）: {result['mw_ds']:,.2f} g/mol
🧪 濃度: {result['conc']:.3f} ng/µL
🧾 コピー数: {result['copies']:.3e} copies/µL
""")
        except Exception as e:
            print(f"⚠️ エラー: {e}")

def on_clear_clicked(b):
    sequence_input.value = ''
    concentration_input.value = 0.0
    with output_area:
        clear_output()

run_button.on_click(on_run_clicked)
clear_button.on_click(on_clear_clicked)

# ===== 表示 =====
display(HTML(custom_css))
display(widgets.HTML("<h3 style='font-size: 24px;'>🧬 Circular dsDNA コピー数計算ツール</h3>"))
display(sequence_input)
display(concentration_input)
display(widgets.HBox([run_button, clear_button]))
display(output_area)


HTML(value="<h3 style='font-size: 24px;'>🧬 Circular dsDNA コピー数計算ツール</h3>")

Textarea(value='', description='配列:', layout=Layout(height='160px', width='100%'), placeholder="ここにプラスミド配列（5'→…

FloatText(value=0.0, description='ng/µL:', style=DescriptionStyle(description_width='60px'))

Output(layout=Layout(border='1px solid #ccc', height='220px', width='100%'))